# Solutions — enumerate, zip, Unpacking & Slicing

Exercise notebook: `03_pythonic_python/02_enumerate_zip_unpacking.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · Number lines with aligned prefixes

**Working solution**

In [ ]:
def number_lines(lines, start=1):
    width = len(str(start + len(lines) - 1)) if lines else 1
    return [f"{n:0{width}d}: {line}" for n, line in enumerate(lines, start=start)]

**Explanation**

`enumerate(..., start=)` + an f-string with a nested width spec (`{n:0{width}d}`).

**Complexity:** O(n).

**Common mistakes**

- `for i in range(len(lines))` then `lines[i]`.

## Exercise 2 — 🟢 Easy · Header + rows → dicts, strictly

**Working solution**

In [ ]:
def rows_to_dicts(header, rows):
    return [dict(zip(header, row, strict=True)) for row in rows]

**Explanation**

`dict(zip(keys, values))` is the idiom. Plain `zip` silently truncates to the shortest input — a nasty data-loss bug; `strict=True` raises.

**Complexity:** O(n × cols).

**Common mistakes**

- Plain `zip` → missing columns silently dropped.

## Exercise 3 — 🟡 Medium · Deltas between consecutive meter readings

**Working solution**

In [ ]:
from itertools import pairwise

def reading_deltas(readings):
    result = []
    for (_, prev), (ts, curr) in pairwise(readings):
        delta = curr - prev
        result.append((ts, round(curr if delta < 0 else delta, 3)))
    return result

**Explanation**

`pairwise` (3.10+) yields overlapping pairs; nested tuple unpacking in the `for` header names every piece. Pre-3.10: `zip(readings, readings[1:])`.

**Complexity:** O(n).

**Common mistakes**

- `for i in range(1, len(r))` with `r[i][1] - r[i-1][1]` — correct, but four indexing operations to read.

## Exercise 4 — 🟠 Hard · Row-oriented ↔ column-oriented

**Working solution**

In [ ]:
def to_columnar(records):
    columns = list(dict.fromkeys(k for r in records for k in r))
    return {c: [r.get(c) for r in records] for c in columns}

def from_columnar(columns):
    names = list(columns)
    return [dict(zip(names, values)) for values in zip(*columns.values(), strict=True)]

**Explanation**

`zip(*columns.values())` transposes: it takes the i-th element of every column as one row. `dict.fromkeys` over a generator
gives an ordered union of keys.

**Complexity:** O(n × cols).

**Common mistakes**

- Using `set()` for the key union → column order becomes random.

## Challenge — 🔴 Very Hard · Fixed-width mainframe file parser

**Working solution**

In [ ]:
from itertools import accumulate

def parse_fixed_width(lines, spec):
    widths = [w for _, w, _ in spec]
    total = sum(widths)
    starts = [0, *accumulate(widths)][:-1]
    records = []
    for line_no, line in enumerate(lines, start=1):
        if len(line) > total:
            raise ValueError(f"line {line_no}: length {len(line)} exceeds {total}")
        line = line.ljust(total)
        record = {}
        for (name, width, convert), start in zip(spec, starts):
            if name == "_":
                continue
            raw = line[start:start + width].strip()
            record[name] = convert(raw) if raw else None
        records.append(record)
    return records

**Explanation**

`[0, *accumulate(widths)][:-1]` turns widths into start offsets — a running sum, no manual counter. Iterating
`zip(spec, starts)` with nested unpacking `(name, width, convert), start` keeps every piece named.

**Complexity:** O(lines × fields).

**Common mistakes**

- Hard-coding offsets that drift when the spec changes.
- Not stripping padding, so `'INR  '` ≠ `'INR'`.